# 00 - 数据集探索与可视化

本 Notebook 完成以下内容：
1. 加载 ORL (AT&T) 人脸数据集
2. 查看数据基本统计
3. 展示样本图像
4. 分析类内/类间差异（为双向聚类提供直觉）
5. 查看均值脸

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.family'] = 'DejaVu Sans'

In [2]:
from data.load_dataset import load_orl, train_test_split_orl, dataset_info

# 加载 ORL 数据集（需先将数据放入 ../data/ORL/ 目录）
X, y = load_orl(data_dir='../data/ORL')
dataset_info(X, y, name='ORL')

[ORL]
  Samples   : 400
  Features  : 4096
  Classes   : 40
  Per class : 10
  Value range: [0.000, 1.000]


In [3]:
from src.utils import show_faces

# ORL 图像尺寸：92 x 112（宽 x 高）
IMG_SHAPE = (64, 64)

fig = show_faces(X, y, img_shape=IMG_SHAPE,
                  n_subjects=6, n_per_subject=4,
                  title='ORL Database — Sample Images')
plt.show()

C:\Users\86175\AppData\Local\Temp\ipykernel_35876\77427283.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
# 均值脸（全局）
mean_face = X.mean(axis=0)

fig, axes = plt.subplots(1, 3, figsize=(10, 4))

axes[0].imshow(mean_face.reshape(IMG_SHAPE), cmap='gray')
axes[0].set_title('全局均值脸')
axes[0].axis('off')

# 各类均值脸（前 5 类）
for i, cls in enumerate(range(5)):
    cls_mean = X[y == cls].mean(axis=0)
    if i == 0:
        axes[1].imshow(cls_mean.reshape(IMG_SHAPE), cmap='gray')
        axes[1].set_title(f'类 {cls} 均值脸')
        axes[1].axis('off')

# 像素值分布
axes[2].hist(X.ravel(), bins=50, color='steelblue', edgecolor='white')
axes[2].set_title('像素值分布')
axes[2].set_xlabel('Pixel Value')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../results/figures/00_data_overview.png', dpi=150, bbox_inches='tight')
plt.show()

C:\Users\86175\AppData\Local\Temp\ipykernel_35876\2513984406.py:24: UserWarning: Glyph 20840 (\N{CJK UNIFIED IDEOGRAPH-5168}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\86175\AppData\Local\Temp\ipykernel_35876\2513984406.py:24: UserWarning: Glyph 23616 (\N{CJK UNIFIED IDEOGRAPH-5C40}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\86175\AppData\Local\Temp\ipykernel_35876\2513984406.py:24: UserWarning: Glyph 22343 (\N{CJK UNIFIED IDEOGRAPH-5747}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\86175\AppData\Local\Temp\ipykernel_35876\2513984406.py:24: UserWarning: Glyph 20540 (\N{CJK UNIFIED IDEOGRAPH-503C}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\86175\AppData\Local\Temp\ipykernel_35876\2513984406.py:24: UserWarning: Glyph 33080 (\N{CJK UNIFIED IDEOGRAPH-8138}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\86175\AppData\Local\Temp\ipykernel_35876\2513984406.py:24: UserWarning: Glyph 31867 (\

C:\Users\86175\AppData\Local\Temp\ipykernel_35876\2513984406.py:25: UserWarning: Glyph 31867 (\N{CJK UNIFIED IDEOGRAPH-7C7B}) missing from font(s) DejaVu Sans.
  plt.savefig('../results/figures/00_data_overview.png', dpi=150, bbox_inches='tight')
C:\Users\86175\AppData\Local\Temp\ipykernel_35876\2513984406.py:25: UserWarning: Glyph 20687 (\N{CJK UNIFIED IDEOGRAPH-50CF}) missing from font(s) DejaVu Sans.
  plt.savefig('../results/figures/00_data_overview.png', dpi=150, bbox_inches='tight')
C:\Users\86175\AppData\Local\Temp\ipykernel_35876\2513984406.py:25: UserWarning: Glyph 32032 (\N{CJK UNIFIED IDEOGRAPH-7D20}) missing from font(s) DejaVu Sans.
  plt.savefig('../results/figures/00_data_overview.png', dpi=150, bbox_inches='tight')
C:\Users\86175\AppData\Local\Temp\ipykernel_35876\2513984406.py:25: UserWarning: Glyph 20998 (\N{CJK UNIFIED IDEOGRAPH-5206}) missing from font(s) DejaVu Sans.
  plt.savefig('../results/figures/00_data_overview.png', dpi=150, bbox_inches='tight')
C:\Users\861

C:\Users\86175\AppData\Local\Temp\ipykernel_35876\2513984406.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# 类内与类间距离分析
from sklearn.metrics import pairwise_distances

# 计算所有样本对的欧氏距离
D = pairwise_distances(X, metric='euclidean')

intra_dists = []
inter_dists = []

n = len(y)
for i in range(n):
    for j in range(i+1, n):
        if y[i] == y[j]:
            intra_dists.append(D[i,j])
        else:
            inter_dists.append(D[i,j])

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(intra_dists, bins=40, alpha=0.6, label='类内距离', color='#4c72b0')
ax.hist(inter_dists, bins=40, alpha=0.4, label='类间距离', color='#dd8452')
ax.set_xlabel('Euclidean Distance')
ax.set_ylabel('Count')
ax.set_title('ORL 数据集 — 类内 vs 类间距离分布')
ax.legend()
plt.tight_layout()
plt.savefig('../results/figures/00_dist_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'类内距离均值: {np.mean(intra_dists):.4f}')
print(f'类间距离均值: {np.mean(inter_dists):.4f}')
print(f'类间/类内比: {np.mean(inter_dists)/np.mean(intra_dists):.2f}')

C:\Users\86175\AppData\Local\Temp\ipykernel_35876\1006758277.py:25: UserWarning: Glyph 25968 (\N{CJK UNIFIED IDEOGRAPH-6570}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\86175\AppData\Local\Temp\ipykernel_35876\1006758277.py:25: UserWarning: Glyph 25454 (\N{CJK UNIFIED IDEOGRAPH-636E}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\86175\AppData\Local\Temp\ipykernel_35876\1006758277.py:25: UserWarning: Glyph 38598 (\N{CJK UNIFIED IDEOGRAPH-96C6}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\86175\AppData\Local\Temp\ipykernel_35876\1006758277.py:25: UserWarning: Glyph 31867 (\N{CJK UNIFIED IDEOGRAPH-7C7B}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\86175\AppData\Local\Temp\ipykernel_35876\1006758277.py:25: UserWarning: Glyph 20869 (\N{CJK UNIFIED IDEOGRAPH-5185}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\86175\AppData\Local\Temp\ipykernel_35876\1006758277.py:25: UserWarning: Glyph 38388 (\

类内距离均值: 8.3786
类间距离均值: 12.4147
类间/类内比: 1.48


In [6]:
# 划分训练集/测试集
X_train, X_test, y_train, y_test = train_test_split_orl(X, y, n_train=5)

print(f'训练集: {X_train.shape}, 测试集: {X_test.shape}')
print(f'训练标签分布: {dict(zip(*np.unique(y_train, return_counts=True)))}')

训练集: (200, 4096), 测试集: (200, 4096)
训练标签分布: {np.int64(0): np.int64(5), np.int64(1): np.int64(5), np.int64(2): np.int64(5), np.int64(3): np.int64(5), np.int64(4): np.int64(5), np.int64(5): np.int64(5), np.int64(6): np.int64(5), np.int64(7): np.int64(5), np.int64(8): np.int64(5), np.int64(9): np.int64(5), np.int64(10): np.int64(5), np.int64(11): np.int64(5), np.int64(12): np.int64(5), np.int64(13): np.int64(5), np.int64(14): np.int64(5), np.int64(15): np.int64(5), np.int64(16): np.int64(5), np.int64(17): np.int64(5), np.int64(18): np.int64(5), np.int64(19): np.int64(5), np.int64(20): np.int64(5), np.int64(21): np.int64(5), np.int64(22): np.int64(5), np.int64(23): np.int64(5), np.int64(24): np.int64(5), np.int64(25): np.int64(5), np.int64(26): np.int64(5), np.int64(27): np.int64(5), np.int64(28): np.int64(5), np.int64(29): np.int64(5), np.int64(30): np.int64(5), np.int64(31): np.int64(5), np.int64(32): np.int64(5), np.int64(33): np.int64(5), np.int64(34): np.int64(5), np.int64(35): np.int6